## Bronze to Silver: Data Cleaning and Transformation for Dimension Tables


In [1]:
import duckdb
import pandas as pd

In [12]:
# Connect database
try:
    con = duckdb.connect("../data/finance_analytics_light_raw.duckdb")
    print("✅Successfully connected to the database")
except ValueError as e:
    print(f"Failed connected to the database{e}")

✅Successfully connected to the database


#### Show Schema


In [4]:
# Show All Schema
con.sql(
    """
    SHOW SCHEMAS
    """
)

┌─────────────────────────────┬─────────────┬─────────┐
│        database_name        │ schema_name │ current │
│           varchar           │   varchar   │ boolean │
├─────────────────────────────┼─────────────┼─────────┤
│ finance_analytics_light_raw │ bronze      │ false   │
│ finance_analytics_light_raw │ gold        │ false   │
│ finance_analytics_light_raw │ main        │ true    │
│ finance_analytics_light_raw │ silver      │ false   │
└─────────────────────────────┴─────────────┴─────────┘

In [8]:
# Show all tables from bronze schema
con.sql(
    """
    SHOW TABLES FROM finance_analytics_light_raw.bronze
    """
)

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ account_mapping │
│ accounts        │
│ geography       │
│ gl_transactions │
│ stores          │
└─────────────────┘

#### Accounts


In [9]:
# Check bronze.accounts table
con.sql(
    """
    SELECT
    *
    FROM bronze.accounts
    """
)

┌────────────────┬────────────────────┬───────────────────┬──────────┐
│ account_number │    account_name    │   account_type    │ currency │
│    varchar     │      varchar       │      varchar      │ varchar  │
├────────────────┼────────────────────┼───────────────────┼──────────┤
│ 3000           │ Product Sales      │ Revenue           │ NOK      │
│ 4000           │ Cost of Goods Sold │ COGS              │ NOK      │
│ 5000           │ Salaries & Wages   │ Operating Expense │ NOK      │
│ 5100           │ Rent Expense       │ Operating Expense │ NOK      │
│ 5200           │ Marketing Expense  │ Operating Expense │ NOK      │
│ 7000           │ Interest Expense   │ Financial         │ NOK      │
└────────────────┴────────────────────┴───────────────────┴──────────┘

In [13]:
# Create silver accounts table
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE silver.accounts AS 
        SELECT
            CAST(account_number AS INTEGER)     AS account_number,
            TRIM(account_name)                  AS account_name,
            UPPER(TRIM(account_type))           AS account_type,
            UPPER(TRIM(currency))               AS currency,
        FROM bronze.accounts
        """
    )
    print("✅ Created silver accounts table")
except ValueError as error:
    print(f"Failed create silver accounts table, {error}")

✅ Created silver accounts table


In [14]:
# Check silver accounts table
con.sql(
    """
    SELECT
    *
    FROM silver.accounts
    """
)

┌────────────────┬────────────────────┬───────────────────┬──────────┐
│ account_number │    account_name    │   account_type    │ currency │
│     int32      │      varchar       │      varchar      │ varchar  │
├────────────────┼────────────────────┼───────────────────┼──────────┤
│           3000 │ Product Sales      │ REVENUE           │ NOK      │
│           4000 │ Cost of Goods Sold │ COGS              │ NOK      │
│           5000 │ Salaries & Wages   │ OPERATING EXPENSE │ NOK      │
│           5100 │ Rent Expense       │ OPERATING EXPENSE │ NOK      │
│           5200 │ Marketing Expense  │ OPERATING EXPENSE │ NOK      │
│           7000 │ Interest Expense   │ FINANCIAL         │ NOK      │
└────────────────┴────────────────────┴───────────────────┴──────────┘

#### General ledger transactions


In [15]:
# Check bronze gl transactions table
con.sql(
    """
    SELECT
    *
    FROM bronze.gl_transactions
    """
)

┌────────────────┬──────────────────┬────────────┬────────────────┬──────────────┬──────────┬─────────────────┬──────────────────────────────────────────────┐
│ transaction_id │ transaction_date │ store_code │ account_number │ amount_local │ currency │ document_number │                 description                  │
│     int64      │       date       │  varchar   │     int32      │    double    │ varchar  │     varchar     │                   varchar                    │
├────────────────┼──────────────────┼────────────┼────────────────┼──────────────┼──────────┼─────────────────┼──────────────────────────────────────────────┤
│              1 │ 2025-03-12       │ SE001      │           4000 │     -7149.34 │ NOK      │ DOC-2025-000001 │ Cost of Goods Sold - Recently thought month. │
│              2 │ 2025-09-24       │ SE001      │           4000 │     -6023.02 │ NOK      │ DOC-2025-000002 │ Cost of Goods Sold - Per send.               │
│              3 │ 2026-05-23       │ SE001   

In [ ]:
# Create silver gl transactions table
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE silver.gl_transactions AS
        SELECT
            transaction_id,
            CAST(transaction_date AS DATE) AS transaction_date,
            store_code,
            account_number,
            amount_local,
            CASE 
                WHEN amount_local >= 0 THEN 'CREDIT'
                ELSE 'DEBIT'
            END AS dc_flag,
            UPPER(TRIM(currency))                   AS currency,
            document_number,
            TRIM(description)                       AS description
        FROM bronze.gl_transactions
        """
    )
    print("✅ Created silver gl_transactions table")
except ValueError as error:
    print(f"Failed create silver gl_transactions table, {error}")

✅ Created silver gl_transactions table


In [ ]:
# Check silver gl_transactions table
con.sql(
    """
    SELECT
    *
    FROM silver.gl_transactions
    """
)

┌────────────────┬──────────────────┬────────────┬────────────────┬──────────────┬─────────┬──────────┬─────────────────┬──────────────────────────────────────────────┐
│ transaction_id │ transaction_date │ store_code │ account_number │ amount_local │ dc_flag │ currency │ document_number │                 description                  │
│     int64      │       date       │  varchar   │     int32      │    double    │ varchar │ varchar  │     varchar     │                   varchar                    │
├────────────────┼──────────────────┼────────────┼────────────────┼──────────────┼─────────┼──────────┼─────────────────┼──────────────────────────────────────────────┤
│              1 │ 2025-03-12       │ SE001      │           4000 │     -7149.34 │ DEBIT   │ NOK      │ DOC-2025-000001 │ Cost of Goods Sold - Recently thought month. │
│              2 │ 2025-09-24       │ SE001      │           4000 │     -6023.02 │ DEBIT   │ NOK      │ DOC-2025-000002 │ Cost of Goods Sold - Per send.   

#### Stores


In [18]:
# Check bronze stores table
con.sql(
    """
    SELECT
    *
    FROM bronze.stores
    """
)

┌────────────┬───────────────────┬────────────┐
│ store_code │    store_name     │ store_type │
│  varchar   │      varchar      │  varchar   │
├────────────┼───────────────────┼────────────┤
│ NO001      │ NRG Oslo City     │ Store      │
│ NO002      │ NRG Bergen        │ Store      │
│ NO003      │ NRG Trondheim     │ Store      │
│ SE001      │ NRG Stockholm     │ Store      │
│ SE002      │ NRG Gothenburg    │ Store      │
│ EC_NO      │ NRG Online Norway │ Ecom       │
│ EC_SE      │ NRG Online Sweden │ Ecom       │
└────────────┴───────────────────┴────────────┘

In [19]:
# Create silver stores table
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE silver.stores AS 
        SELECT 
            s.store_code,
            TRIM(s.store_name)        AS store_name,
            UPPER(TRIM(s.store_type)) AS store_type,
            g.country,
            g.region
        FROM bronze.stores s
        LEFT JOIN bronze.geography g
            ON s.store_code = g.store_code
        """
    )
    print("✅ Created silver stores table")
except ValueError as error:
    print(f"Failed create silver stores table, {error}")

✅ Created silver stores table


In [20]:
# Check silver stores table
con.sql(
    """
    SELECT 
        *
    FROM silver.stores
    """
)

┌────────────┬───────────────────┬────────────┬─────────┬──────────┐
│ store_code │    store_name     │ store_type │ country │  region  │
│  varchar   │      varchar      │  varchar   │ varchar │ varchar  │
├────────────┼───────────────────┼────────────┼─────────┼──────────┤
│ NO001      │ NRG Oslo City     │ STORE      │ Norway  │ East     │
│ NO002      │ NRG Bergen        │ STORE      │ Norway  │ West     │
│ NO003      │ NRG Trondheim     │ STORE      │ Norway  │ Mid      │
│ SE001      │ NRG Stockholm     │ STORE      │ Sweden  │ East     │
│ SE002      │ NRG Gothenburg    │ STORE      │ Sweden  │ West     │
│ EC_NO      │ NRG Online Norway │ ECOM       │ Norway  │ National │
│ EC_SE      │ NRG Online Sweden │ ECOM       │ Sweden  │ National │
└────────────┴───────────────────┴────────────┴─────────┴──────────┘

#### Account mapping


In [21]:
# Check bronze account_mapping table
con.sql(
    """
    SELECT
        *
    FROM bronze.account_mapping
    """
)

┌───────────────┬────────────────────┬────────────────────┬───────────────┬───────────┬───────────────────────────────────────────────────────────────────┐
│ AccountNumber │    AccountName     │       PLLine       │ StatementType │ SortOrder │                               Notes                               │
│     int64     │      varchar       │      varchar       │    varchar    │   int64   │                              varchar                              │
├───────────────┼────────────────────┼────────────────────┼───────────────┼───────────┼───────────────────────────────────────────────────────────────────┤
│          3000 │ Product Sales      │ Revenue            │ P&L           │        10 │ NULL                                                              │
│          3010 │ Online Sales       │ Revenue            │ P&L           │        10 │ New online channel – mapping not yet confirmed                    │
│          4000 │ Cost of Goods Sold │ COGS               │ P&L 

In [22]:
# Create silver account_mapping table
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE silver.account_mapping AS
        SELECT 
            CAST(AccountNumber AS INTEGER)  AS account_number,
            TRIM(AccountName)               AS account_name,
            TRIM(PLLine)                    AS pl_line,
            UPPER(TRIM(StatementType))      AS statement_type,
            CAST(SortOrder AS INTEGER)      AS sort_order,
            TRIM(Notes)                     AS notes
        FROM bronze.account_mapping
        """
    )
    print("✅ Created silver account mapping table")
except ValueError as error:
    print(f"Failed create silver table, {error}")

✅ Created silver account mapping table


In [23]:
# Check silver account_mapping table
con.sql(
    """
    SELECT
        *
    FROM silver.account_mapping
    """
)

┌────────────────┬────────────────────┬────────────────────┬────────────────┬────────────┬───────────────────────────────────────────────────────────────────┐
│ account_number │    account_name    │      pl_line       │ statement_type │ sort_order │                               notes                               │
│     int32      │      varchar       │      varchar       │    varchar     │   int32    │                              varchar                              │
├────────────────┼────────────────────┼────────────────────┼────────────────┼────────────┼───────────────────────────────────────────────────────────────────┤
│           3000 │ Product Sales      │ Revenue            │ P&L            │         10 │ NULL                                                              │
│           3010 │ Online Sales       │ Revenue            │ P&L            │         10 │ New online channel – mapping not yet confirmed                    │
│           4000 │ Cost of Goods Sold │ COGS  

In [24]:
con.close()